In [9]:
import re 
import string 
import nltk 
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

#nltk resources 
nltk.download('stopwords' , quiet = True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


#function 

def preprocess_review(raw_reviewe: str) -> str:
    text = raw_reviewe.lower()
    text = text.translate(str.maketrans('' , '' ,string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip() 

    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in STOP_WORDS]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return ' '.join(tokens)   


sample_reviews = [
    "The food arrived VERY late and it was completely cold!! Really disappointed :(",
    "Amazing service, the delivery guy was super polite... Loved the biryani, will order again!!",
    "App kept crashing while I tried to track my order, and the support team never responded.",
]

for i , review in enumerate(sample_reviews , 1):
    cleaned = preprocess_review(review)
    print(f"Sample {i}")
    print(f"  Original: {review}")
    print(f"  Cleaned : {cleaned}")
    print()

Sample 1
  Original: The food arrived VERY late and it was completely cold!! Really disappointed :(
  Cleaned : food arrived late completely cold really disappointed

Sample 2
  Original: Amazing service, the delivery guy was super polite... Loved the biryani, will order again!!
  Cleaned : amazing service delivery guy super polite loved biryani order

Sample 3
  Original: App kept crashing while I tried to track my order, and the support team never responded.
  Cleaned : app kept crashing tried track order support team never responded



In [10]:
#Task 2 

import pandas as pd 
from sklearn.feature_extraction.text import TfidfVectorizer

reviews = [
    "The biryani was delicious and arrived hot",
    "Food was cold and delivery took too long",
    "Average taste, nothing special about this pizza",
    "Amazing service and the burger was tasty",
    "Order was late and packaging was damaged",
    "The pasta was okay, not great but not bad",
    "Best butter chicken I have ever tasted",
    "Terrible experience, food arrived cold",
]


tfidf = TfidfVectorizer()
matrix = tfidf.fit_transform(reviews)

words = tfidf.get_feature_names_out()
df = pd.DataFrame(matrix.toarray(), columns=words)

print(df.round(2))

for i in range(len(reviews)):
    top3 = df.iloc[i].sort_values(ascending=False).head(3)
    print(f"\nReview {i+1}: {reviews[i]}")
    print(top3)


   about  amazing   and  arrived  average  ...   the  this  too  took   was
0   0.00     0.00  0.29     0.38     0.00  ...  0.33  0.00  0.0   0.0  0.25
1   0.00     0.00  0.26     0.00     0.00  ...  0.00  0.00  0.4   0.4  0.23
2   0.38     0.00  0.00     0.00     0.38  ...  0.00  0.38  0.0   0.0  0.00
3   0.00     0.44  0.28     0.00     0.00  ...  0.32  0.00  0.0   0.0  0.25
4   0.00     0.00  0.27     0.00     0.00  ...  0.00  0.00  0.0   0.0  0.47
5   0.00     0.00  0.00     0.00     0.00  ...  0.23  0.00  0.0   0.0  0.18
6   0.00     0.00  0.00     0.00     0.00  ...  0.00  0.00  0.0   0.0  0.00
7   0.00     0.00  0.00     0.41     0.00  ...  0.00  0.00  0.0   0.0  0.00

[8 rows x 42 columns]

Review 1: The biryani was delicious and arrived hot
hot          0.449809
biryani      0.449809
delicious    0.449809
Name: 0, dtype: float64

Review 2: Food was cold and delivery took too long
took    0.404166
too     0.404166
long    0.404166
Name: 1, dtype: float64

Review 3: Average tast

In [12]:
import re
import string
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

complaints = [
    # Delivery
    ("My order is taking way too long to arrive", "Delivery"),
    ("The delivery partner never showed up at my address", "Delivery"),
    ("Order was delivered to the wrong address", "Delivery"),
    ("Delivery has been delayed by more than an hour", "Delivery"),
    ("The rider cancelled my order without any reason", "Delivery"),
    ("I have been waiting for my delivery for two hours", "Delivery"),
    ("The delivery boy could not find my location", "Delivery"),
    ("My package was left outside instead of handed to me", "Delivery"),
    ("Order status still shows out for delivery since morning", "Delivery"),
    ("The delivery was extremely late during peak hours", "Delivery"),

    # Food Quality
    ("The food arrived completely cold and tasteless", "Food Quality"),
    ("There was a hair found in my pasta", "Food Quality"),
    ("The pizza was soggy and undercooked", "Food Quality"),
    ("My order had a strange smell and tasted spoiled", "Food Quality"),
    ("The portion size was way smaller than usual", "Food Quality"),
    ("The chicken was raw in the middle", "Food Quality"),
    ("Food quality has become really poor recently", "Food Quality"),
    ("The dessert was melted and inedible on arrival", "Food Quality"),
    ("The curry tasted completely different from usual", "Food Quality"),
    ("Received stale bread with my sandwich order", "Food Quality"),

    # App
    ("The app keeps crashing whenever I try to checkout", "App"),
    ("I cannot log into my account on the app", "App"),
    ("Payment failed but amount was deducted from my card", "App"),
    ("The app is not showing my current order status", "App"),
    ("Coupon code is not applying at checkout", "App"),
    ("The app freezes every time I open the menu", "App"),
    ("Notifications are not working properly on the app", "App"),
    ("The tracking map is not updating in the app", "App"),
    ("App shows an error every time I place an order", "App"),
    ("I am unable to update my delivery address in the app", "App"),
]


df_data = pd.DataFrame(complaints , columns=["text" , "label"])

#preprocess text 
df_data["cleaned"]= df_data["text"].apply(preprocess_review)

#convert to tf-idf 
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df_data["cleaned"])
y = df_data["label"]

#train-test 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#model 
model = MultinomialNB()
model.fit(X_train, y_train)


y_pred = model.predict(X_test)

#  Evaluate
print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
print(pd.DataFrame(cm, index=labels, columns=labels))

Classification Report:
              precision    recall  f1-score   support

         App       1.00      1.00      1.00         2
    Delivery       1.00      1.00      1.00         2
Food Quality       1.00      1.00      1.00         2

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6

Confusion Matrix:
              App  Delivery  Food Quality
App             2         0             0
Delivery        0         2             0
Food Quality    0         0             2


In [18]:
#Task - 4
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv("food_delivery_sentiment.csv")
df["cleaned_review"] = df["review"].apply(preprocess_review)

X = df["cleaned_review"]
y = df["sentiment"]

#train-test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#build 2 piplines objects 

nb_pipeline = Pipeline([
    ("tfidf" , TfidfVectorizer()),
    ("classifier", MultinomialNB())
])

lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", LogisticRegression(max_iter=1000))
])

#train both pipeline 
nb_pipeline.fit(X_train, y_train)
lr_pipeline.fit(X_train, y_train)

results = {}
for name, pipeline in [("Naive Bayes", nb_pipeline), ("Logistic Regression", lr_pipeline)]:
    y_pred = pipeline.predict(X_test)
    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, pos_label="Positive"),
        "Recall": recall_score(y_test, y_pred, pos_label="Positive"),
        "F1-score": f1_score(y_test, y_pred, pos_label="Positive"),
    }

comparison_df = pd.DataFrame(results).round(3)
print("Model Comparison:")
print(comparison_df) 




Model Comparison:
           Naive Bayes  Logistic Regression
Accuracy         0.786                0.786
Precision        0.833                0.833
Recall           0.714                0.714
F1-score         0.769                0.769
